<a href="https://colab.research.google.com/github/Annette-1/SIMULACION-II/blob/main/anylogic_abejas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <span style="color:magenta;"> **Modelo en AnyLogic: población de abejas y recursos florales** </span>

<span style="color:teal;">**Nombre:** </span> Jerónimo López Annette Michelle

<span style="color:magenta;">**Objetivo:**</span>
Modelo híbrido (dinámica de sistemas + agentes) de una población de abejas que depende de la cantidad de flores disponibles. Se compara la solución de dinámica de sistemas de AnyLogic con la solución analítica y con la versión estocástica basada en agentes.

**Unidad de tiempo:** 1 mes.  
**Herramienta del modelo:** AnyLogic Personal Learning Edition (archivo `.alp` incluido en este repositorio).  
**Autor:** _nombre, materia y fecha_

## <span style="color:teal;">**Descripción del modelo**</span>

La población de abejas sigue un crecimiento logístico cuya capacidad de carga depende de las flores:

$$\frac{dA}{dt} = rA - \frac{rA^2}{K}, \qquad K = k\,F(t)$$

Las flores pueden mantenerse constantes o disminuir con el tiempo (pérdida de recursos florales):

$$F(t) = F_\infty + (F_0 - F_\infty)e^{-\lambda t} \quad \text{si } perdida = true, \qquad F(t) = F_0 \quad \text{si } perdida = false$$

Con $K$ constante, la solución analítica es:

$$A(t) = \frac{K}{1 + \dfrac{K - A_0}{A_0}\,e^{-rt}}$$

## <span style="color:teal;">**Parámetros**</span>

| Parámetro | Tipo | Valor | Significado |
|---|---|---|---|
| `r` | double | 0.8 | Tasa de crecimiento de las abejas |
| `k` | double | 10 | Abejas que soporta cada flor |
| `F0` | int | 100 | Flores iniciales |
| `Finf` | double | 40 | Flores finales si hay pérdida |
| `lam` | double | 0.25 | Velocidad de pérdida de flores |
| `A0` | int | 50 | Abejas iniciales |
| `dt` | double | 0.05 | Paso de tiempo de los agentes (meses) |
| `perdida` | boolean | true | Activa o desactiva la pérdida de flores |

## <span style="color:teal;">**Estructura del modelo en AnyLogic**</span>

**Dinámica de sistemas (en `Main`)**
- Variable dinámica `F`: número de flores en el tiempo.
- Variable dinámica `Kcap = k·F`: capacidad de carga.
- Stock `AbejasSD` (valor inicial `A0`).
- Flujos: `Nacimientos = r·AbejasSD` y `muertes = r·AbejasSD²/Kcap`.
- Solucionador: ecuaciones diferenciales **RK4**.

**Agentes**
- Poblaciones `flores` (tipo `Flor`, inicial `F0`) y `abejas` (tipo `Abeja`, inicial `A0`), distribuidas al azar en el espacio.
- Evento `pasoEscenario` (en `Main`, cíclico cada `dt`): elimina flores al azar hasta alcanzar el número objetivo.
- Evento `vida` (en `Abeja`, cíclico cada `dt`): cada abeja puede generar una nueva (probabilidad `r·dt`) y puede morir con probabilidad `r·A/K·dt`.
- Evento `mover` (en `Abeja`, cíclico cada 0.5): la abeja se desplaza a un punto aleatorio del terreno.

**Animación**
- Ventana 3D con terreno, flores y abejas, con cámara fija sobre el terreno.
- Textos dinámicos (mes, número de flores y de abejas).
- Gráficas de tipo Time Plot: abejas (agentes y SD), capacidad `Kcap` y comparación con la solución analítica.

### 5.1 Sin pérdida de flores: RK4 contra solución analítica

Con `perdida = false`, $K = kF_0$ es constante y ambas soluciones deben coincidir.

### 5.2 Con pérdida de flores

Ahora $K(t) = kF(t)$ disminuye. La solución analítica con $K$ constante ya no aplica, pero la integración numérica sí sigue a la capacidad cambiante.

### 5.3 Simulación estocástica (reglas de los agentes)

En cada paso `dt`, cada abeja se reproduce con probabilidad `r·dt` y muere con probabilidad `min(1, r·A/K·dt)`, igual que el evento `vida` del modelo. Como el proceso es aleatorio, la población fluctúa alrededor de la curva determinista. Las fluctuaciones se notan más con poblaciones pequeñas, por eso aquí se muestra también un caso con `k` bajo.

## 6. Resultados y conclusiones

Valores de referencia del modelo en AnyLogic con los parámetros de la tabla:

| Escenario | Resultado de `AbejasSD` |
|---|---|
| `perdida = false` | Tiende a ≈ 1000 (K = k·F0); alcanza el 90 % de K cerca de t = 6.5 |
| `perdida = true` | Tiende a ≈ 406; `Kcap(20)` = 404 |

- Con `perdida = false`, la curva de dinámica de sistemas de AnyLogic coincide con la **solución analítica** de la ecuación logística, lo que valida la implementación del modelo.
- Con `perdida = true`, K disminuye con el tiempo; la solución analítica con K constante deja de aplicar, pero la dinámica de sistemas sí sigue a la capacidad cambiante.
- El modelo de **agentes** es estocástico: la población fluctúa alrededor de la curva de dinámica de sistemas. Las oscilaciones se aprecian mejor con poblaciones pequeñas (valores bajos de `k`).
- El modelo logístico por sí solo no produce oscilaciones periódicas; solo se acerca a K de forma suave. Las variaciones que se ven provienen de la aleatoriedad de los agentes.